# Chapter 4 — Linear Transformations and Matrix Algebra: SageMath Companion

Symbolic, exact-arithmetic counterpart to `code/python/04_linear_transformations.ipynb`. Where the Python notebook used NumPy floats, this notebook uses SageMath's `Matrix(QQ, …)` and `Matrix(SR, …)` for exact and symbolic arithmetic — rotations with an *unevaluated* `θ`, inverse formulas derived rather than guessed, and compositions proven (not just sampled).

**To run this notebook:** open in [CoCalc](https://cocalc.com/) (no install needed), or locally with the SageMath kernel (`sage -n jupyter`).

**Kernel:** SageMath.

**Layout:**

1. Linearity — symbolic check of additivity and homogeneity
2. Building the standard matrix from `T(e_i)`
3. The 2D catalog with a *symbolic* rotation angle
4. Composition = matrix multiplication; non-commutativity, symbolic
5. The `2 × 2` inverse formula — derived, not stated
6. General inverse via Gauss–Jordan on `[A | I]`
7. `(AB)^{-1} = B^{-1} A^{-1}` — proved for a symbolic example
8. A rotation is its own inverse with `θ → -θ`
9. **Exercises** for you to solve
10. Solutions

## 1. Linearity — symbolic check

Given `T(x, y) = (3x - y, 2x + 5y)`, confirm both linearity properties hold *identically* (for all `u`, `v`, `c`), not just at sampled points.

In [ ]:
var('u1 u2 v1 v2 c')

def T(vec):
    x, y = vec[0], vec[1]
    return vector([3*x - y, 2*x + 5*y])

u = vector([u1, u2])
v = vector([v1, v2])

lhs_add = T(u + v)
rhs_add = T(u) + T(v)
show('T(u+v) - (T(u) + T(v)) =', (lhs_add - rhs_add).simplify_full())

lhs_hom = T(c * u)
rhs_hom = c * T(u)
show('T(c u) - c T(u)        =', (lhs_hom - rhs_hom).simplify_full())

## 2. The standard matrix

From §4.4: columns of the standard matrix are `T(e_j)`. Let's derive it for the `T` above, then re-derive from scratch for a symbolic rotation.

In [ ]:
A = column_matrix([T(vector([1, 0])), T(vector([0, 1]))])
show('Standard matrix of T:'); show(A)
# Check: A · (u1, u2) matches T(u1, u2)
show('A · (u1, u2) =', A * vector([u1, u2]))
show('T (u1, u2)   =', T(vector([u1, u2])))

## 3. The 2D catalog — symbolic rotation

Build the rotation matrix `R(θ)` with `θ` left symbolic. Sage never picks a numeric value, so we can *prove* facts like `R(α) R(β) = R(α + β)`.

In [ ]:
var('theta alpha beta')

def R(th):
    return matrix(SR, [[cos(th), -sin(th)],
                       [sin(th),  cos(th)]])

show('R(θ) =', R(theta))

# Angle addition via matrix multiplication — the trig identities fall out
prod = (R(alpha) * R(beta)).apply_map(lambda expr: expr.trig_simplify())
show('R(α) · R(β) =', prod)
show('R(α + β)   =', R(alpha + beta))
show('Equal?     ', bool(prod == R(alpha + beta)))

In [ ]:
# Explicit catalog in Sage — each derived as column_matrix([T(e1), T(e2)])
var('k k1 k2')

catalog = {
    'Identity':           matrix(SR, [[1, 0], [0, 1]]),
    'Uniform scaling k':  matrix(SR, [[k, 0], [0, k]]),
    'Non-uniform k1, k2': matrix(SR, [[k1, 0], [0, k2]]),
    'Rotate θ':           R(theta),
    'Reflect x-axis':     matrix(SR, [[1, 0], [0, -1]]),
    'Reflect y=x':        matrix(SR, [[0, 1], [1, 0]]),
    'Horizontal shear k': matrix(SR, [[1, k], [0, 1]]),
    'Project onto x':     matrix(SR, [[1, 0], [0, 0]]),
}

for name, M in catalog.items():
    show(name + ':'); show(M)

## 4. Composition is non-commutative — symbolically

With a symbolic rotation and a shear, we can *see* `A B ≠ B A` as two genuinely different symbolic matrices.

In [ ]:
A = R(theta)
B = matrix(SR, [[1, k], [0, 1]])     # horizontal shear

AB = (A * B).apply_map(lambda e: e.expand())
BA = (B * A).apply_map(lambda e: e.expand())
show('A · B =', AB)
show('B · A =', BA)
show('(A·B) - (B·A) =', (AB - BA).apply_map(lambda e: e.simplify_full()))

## 5. The `2 × 2` inverse formula — derived from scratch

Let `A = [[a, b], [c, d]]` be a generic `2 × 2` matrix. We want `X = [[x, y], [z, w]]` with `A · X = I`. That's four equations in four unknowns — solve symbolically.

In [ ]:
var('a b c d x y z w')
A = matrix(SR, [[a, b], [c, d]])
X = matrix(SR, [[x, y], [z, w]])

eqs = (A * X - matrix.identity(2)).list()   # 4 equations == 0
show('Equations to solve:', eqs)

sol = solve(eqs, x, y, z, w, solution_dict=True)
show('Solution:', sol)

A_inv_formula = X.subs(sol[0])
show('A^{-1} =', A_inv_formula.simplify_full())
show('det A  =', A.det())
show('Rewriting with (a d − b c) in the denominator:',
     (1/(a*d - b*c)) * matrix(SR, [[d, -b], [-c, a]]))

## 6. General inverse via Gauss–Jordan on `[A | I]`

Exact rational arithmetic — no floating-point roundoff. We use the same `add_multiple_of_row` / `rescale_row` helpers as Chapter 3.

In [ ]:
def invert_via_gj(A):
    '''Return A^{-1} (as a Sage matrix) by running Gauss-Jordan on [A | I].'''
    A = matrix(QQ, A)
    n = A.nrows()
    assert A.ncols() == n, 'matrix must be square'
    M = A.augment(matrix.identity(QQ, n))
    for col in range(n):
        # Find a row at or below `col` with a nonzero entry in column `col`
        pivot_row = None
        for r in range(col, n):
            if M[r, col] != 0:
                pivot_row = r; break
        if pivot_row is None:
            raise ValueError('matrix is singular')
        if pivot_row != col:
            M.swap_rows(col, pivot_row)
        M.rescale_row(col, 1 / M[col, col])
        for r in range(n):
            if r != col and M[r, col] != 0:
                M.add_multiple_of_row(r, col, -M[r, col])
    return M.submatrix(0, n, n, n)

A = matrix(QQ, [[1, 0, 2],
                [2, 1, 3],
                [1, 0, 1]])
Ainv = invert_via_gj(A)
show('A^{-1} via Gauss-Jordan:'); show(Ainv)
show('Sage built-in A.inverse():'); show(A.inverse())
show('A · A^{-1} =', A * Ainv)

## 7. `(A B)^{-1} = B^{-1} A^{-1}` — symbolic proof

Take two symbolic invertible `2 × 2` matrices. Show the identity holds.

In [ ]:
var('a b c d p q r s')
A = matrix(SR, [[a, b], [c, d]])
B = matrix(SR, [[p, q], [r, s]])

lhs = (A * B).inverse().apply_map(lambda e: e.simplify_full())
rhs = (B.inverse() * A.inverse()).apply_map(lambda e: e.simplify_full())
show('(A B)^{-1} =', lhs)
show('B^{-1} A^{-1} =', rhs)
show('Equal? ', bool((lhs - rhs).apply_map(lambda e: e.simplify_full()).is_zero()))

## 8. A rotation is its own inverse with `θ → -θ`

So rotation matrices are *orthogonal*: `R(θ)^{-1} = R(θ)^T`. This is a special property that we'll revisit in Chapter 7.

In [ ]:
R_inv_formula = R(-theta).apply_map(lambda e: e.simplify_full())
R_actual_inv  = R(theta).inverse().apply_map(lambda e: e.simplify_full())
show('R(-θ)       =', R_inv_formula)
show('R(θ)^{-1}   =', R_actual_inv)
show('R(θ)^T      =', R(theta).transpose())
show('det R(θ)    =', R(theta).det().simplify_full())

## 9. Exercises

Try each one below — solutions are at the bottom. Don't peek.

**E1.** Prove that the composition of two linear maps is linear. That is, given `S, T: ℝ² → ℝ²` both linear, show `(S ∘ T)(u + v) = (S ∘ T)(u) + (S ∘ T)(v)` and `(S ∘ T)(c u) = c (S ∘ T)(u)`. Do it by direct manipulation using `simplify_full`.

**E2.** Find the matrix that **reflects across the line `y = 2 x`**. (Hint: reflect `e_1` and `e_2` by hand, or use the general formula for reflection across a line through the origin with unit direction `(cos α, sin α)`: `M = [[cos 2α, sin 2α], [sin 2α, -cos 2α]]`.)

**E3.** For `A = matrix(QQ, [[2, 1, 3], [0, 4, 5], [1, 0, 2]])`, find `A^{-1}` by Gauss–Jordan and verify `A · A^{-1} = I`.

**E4.** Show that the product of a rotation `R(θ)` and a reflection across the x-axis is itself a *reflection* — specifically, across the line through the origin at angle `θ / 2`.

**E5.** The **shear group.** Show that the set of horizontal shear matrices `{ [[1, k], [0, 1]] : k ∈ ℝ }` is closed under multiplication and inversion (the product of two horizontal shears is a horizontal shear, and so is the inverse of one). Find the formulas.

In [ ]:
# E1: your code here


In [ ]:
# E2: your code here


In [ ]:
# E3: your code here


In [ ]:
# E4: your code here


In [ ]:
# E5: your code here


## 10. Solutions

In [ ]:
# E1 solution: prove composition of linear maps is linear, symbolically.
var('u1 u2 v1 v2 c')
# S: scale by 2, rotate by pi/3.  T: general linear with entries a, b, c, d (rename to avoid clash).
var('a11 a12 a21 a22 b11 b12 b21 b22')
S = matrix(SR, [[a11, a12], [a21, a22]])
Tm = matrix(SR, [[b11, b12], [b21, b22]])
u = vector([u1, u2]); v = vector([v1, v2])

ST = S * Tm   # standard matrix of S ∘ T

lhs = ST * (u + v)
rhs = ST * u + ST * v
show('(S∘T)(u+v) - (S∘T)u - (S∘T)v =', (lhs - rhs).apply_map(lambda e: e.simplify_full()))

lhs_h = ST * (c * u)
rhs_h = c * (ST * u)
show('(S∘T)(cu) - c·(S∘T)u         =', (lhs_h - rhs_h).apply_map(lambda e: e.simplify_full()))

In [ ]:
# E2 solution: reflection across y = 2x
# Direction vector of line: (1, 2); unit direction: (1, 2)/sqrt(5).
# Angle α with x-axis: tan α = 2, so cos 2α = (1 - tan^2 α)/(1 + tan^2 α) = -3/5,
#                              sin 2α = 2 tan α / (1 + tan^2 α) = 4/5.
M = matrix(QQ, [[-3/5, 4/5], [4/5, 3/5]])
show('Reflection matrix across y = 2x:'); show(M)
# Sanity check: a point on the line should be fixed; a point perpendicular should be negated.
p_on  = vector(QQ, [1, 2])      # on y = 2x
p_off = vector(QQ, [2, -1])     # perpendicular direction (flips to its negative)
show('M · (1, 2)  =', M * p_on,  '  (should equal (1, 2))')
show('M · (2, -1) =', M * p_off, '  (should equal (-2, 1))')
# And M^2 = I (reflecting twice is identity)
show('M^2 =', M^2)

In [ ]:
# E3 solution
A = matrix(QQ, [[2, 1, 3], [0, 4, 5], [1, 0, 2]])
Ainv = invert_via_gj(A)
show('A^{-1} =', Ainv)
show('A · A^{-1} =', A * Ainv)

In [ ]:
# E4 solution: R(θ) · (reflect x-axis) = reflection across y = tan(θ/2) · x
var('theta')
Refl_x = matrix(SR, [[1, 0], [0, -1]])
P = (R(theta) * Refl_x).apply_map(lambda e: e.simplify_full())
show('R(θ) · Refl_x =', P)
# Expected: reflection across line at angle θ/2 from x-axis
Refl_halfangle = matrix(SR, [[cos(theta), sin(theta)],
                             [sin(theta), -cos(theta)]])
show('Reflection across line at angle θ/2 (formula: [[cos 2α, sin 2α], [sin 2α, -cos 2α]] with 2α=θ):',
     Refl_halfangle)
show('Equal? ', bool((P - Refl_halfangle).apply_map(lambda e: e.simplify_full()).is_zero()))
# And reflections square to I
show('P^2 =', (P * P).apply_map(lambda e: e.trig_simplify()))

In [ ]:
# E5 solution: shear group is closed under product and inverse
var('k k1 k2')
S = lambda kk: matrix(SR, [[1, kk], [0, 1]])

prod = S(k1) * S(k2)
show('S(k1) · S(k2) =', prod, '   → S(k1 + k2)')

inv = S(k).inverse()
show('S(k)^{-1}     =', inv, '   → S(-k)')

---

## Where to next

- **Chapter 5 (Sage):** `A.column_space()`, `A.right_kernel()` become the formal **image** and **kernel**. The invertible-matrix theorem gains clauses.
- **Chapter 7 (Sage):** orthogonal matrices — those with `A^{-1} = A^T`, of which rotations are the canonical example.
- **Chapter 8 (Sage):** `det(A)` connects directly to the geometry we saw here — a `2 × 2` matrix is invertible iff `det ≠ 0` iff it doesn't collapse the plane.
- **Chapter 9 (Sage):** eigenvectors — the directions `A` merely *stretches* instead of rotating or shearing.